# KDD: Credit Card Fraud Detection

This notebook follows Selection, Preprocessing, Transformation, Data Mining, and Interpretation/Evaluation.

In [1]:
from google.colab import files
from pathlib import Path
import shutil

data_dir = Path("/content/data")
data_dir.mkdir(exist_ok=True)

uploaded = files.upload()

for filename in uploaded:
    shutil.copy(filename, data_dir / filename)

print("Files in data folder:")
print(list(data_dir.iterdir()))

Saving creditcard.csv to creditcard.csv
Files in data folder:
[PosixPath('/content/data/creditcard.csv')]


In [2]:
import os, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report
DATA_PATH='data/creditcard.csv'
assert os.path.exists(DATA_PATH), 'Download the Kaggle CSV; see ../data/README.md'
df=pd.read_csv(DATA_PATH)
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## Selection and preprocessing

Fraud is rare, so accuracy is not the primary metric. We preserve the class signal and use a stratified split. In production, a chronological split should also be tested.

In [3]:
target='Class'; y=df[target].astype(int); X=df.drop(columns=[target])
print(y.value_counts()); print('fraud rate:', y.mean())
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,stratify=y,random_state=42)
models={'logistic':LogisticRegression(max_iter=1000,class_weight='balanced'),'random_forest':RandomForestClassifier(n_estimators=250,class_weight='balanced',random_state=42,n_jobs=-1)}

Class
0    284315
1       492
Name: count, dtype: int64
fraud rate: 0.001727485630620034


## Transformation and data mining

In [4]:
results=[]
for name,model in models.items():
    pipe=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',model)])
    pipe.fit(X_train,y_train); p=pipe.predict_proba(X_test)[:,1]; pred=(p>=.5).astype(int)
    results.append({'model':name,'pr_auc':average_precision_score(y_test,p),'roc_auc':roc_auc_score(y_test,p)})
    print(name); print(classification_report(y_test,pred,digits=3))
pd.DataFrame(results)

logistic
              precision    recall  f1-score   support

           0      1.000     0.977     0.988     71079
           1      0.062     0.886     0.117       123

    accuracy                          0.977     71202
   macro avg      0.531     0.932     0.553     71202
weighted avg      0.998     0.977     0.987     71202

random_forest
              precision    recall  f1-score   support

           0      1.000     1.000     1.000     71079
           1      0.957     0.732     0.829       123

    accuracy                          0.999     71202
   macro avg      0.978     0.866     0.915     71202
weighted avg      0.999     0.999     0.999     71202



,model,pr_auc,roc_auc
0,logistic,0.703857,0.972638
1,random_forest,0.838239,0.956614


## Interpretation and evaluation

The operational threshold should be chosen from a precision–recall curve using the available analyst capacity. False positives create review cost; false negatives create missed-fraud cost. The final interpretation is an investigation queue, not an automatic account action. Monitor temporal drift and delayed labels.